# 14 — Ensemble Bracket + Predictions Export

Hybrid sim:
- **Group stage**: Reg-DC samples scores (needed for GD/GF tiebreakers)
- **Knockout (R32 → Final)**: MLP+XGB+CB ensemble decides 90-minute outcome; DC handles ET goals

Exports `predictions.json` for the web viz.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import json
import pandas as pd, numpy as np
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.stats import poisson
from src.elo import EloModel
from src.regression_dc import RegressionDixonColesModel
from src.draw_model import match_outcome
from src.bracket import simulate_tournament_48, WC_2026_GROUPS

## Build features on full history

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv  = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])
elo = EloModel()
elo_enriched = elo.fit(df_all)
df = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                  on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])

HOME_ADVANTAGE = 100; FLOOR = 1e5
df['neutral'] = df['neutral'].fillna(False)
df['eff_elo_diff']  = (df['home_elo_pre'] + (~df['neutral']).astype(int)*HOME_ADVANTAGE) - df['away_elo_pre']
df['elo_diff_norm'] = df['eff_elo_diff']/400.0
df['log_value_diff']    = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR)/df['away_top_n_value_eur'].clip(lower=FLOOR))
df['outfield_age_diff'] = df['home_outfield_mean_age'] - df['away_outfield_mean_age']
df['top1_share_diff']   = df['home_top1_share'] - df['away_top1_share']
df['fifa_points_diff']  = (df['home_fifa_points'].fillna(0) - df['away_fifa_points'].fillna(0))/1000.0
# Per-team features for Reg-DC
df['home_elo']  = df['home_elo_pre']/100.0; df['away_elo']  = df['away_elo_pre']/100.0
df['home_logv'] = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR))
df['away_logv'] = np.log10(df['away_top_n_value_eur'].clip(lower=FLOOR))
df['home_fpts'] = df['home_fifa_points'].fillna(0)/1000.0
df['away_fpts'] = df['away_fifa_points'].fillna(0)/1000.0
df['outcome']   = df.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)

FEAT_CLF = ['elo_diff_norm','log_value_diff','outfield_age_diff','top1_share_diff','fifa_points_diff']
RDC_FEAT = ['elo','logv','fpts']
needed = FEAT_CLF + [f'{s}_{c}' for s in ('home','away') for c in RDC_FEAT] + ['outcome']
train = df.dropna(subset=needed)
train = train[(train['home_top_n_value_eur']>FLOOR)&(train['away_top_n_value_eur']>FLOOR)].copy()
print(f'Training on {len(train):,} matches')
train = train.dropna(subset=FEAT_CLF)
print(f'After NaN drop on FEAT_CLF: {len(train):,}')


Training on 7,641 matches
After NaN drop on FEAT_CLF: 7,641


## Fit ensemble on full training set

In [3]:
y = train['outcome'].to_numpy()
X = train[FEAT_CLF].to_numpy()
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

mlp = MLPClassifier(hidden_layer_sizes=(32,16), activation='relu', max_iter=500,
                     learning_rate_init=0.005, alpha=1e-3, early_stopping=True, random_state=0).fit(Xs, y)
xgb_clf = xgb.XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=300, max_depth=4,
                             learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                             eval_metric='mlogloss', n_jobs=4, verbosity=0).fit(X, y)
cb_clf  = CatBoostClassifier(iterations=400, depth=4, learning_rate=0.05, loss_function='MultiClass',
                              verbose=False, random_seed=0, thread_count=4).fit(X, y)
rdc = RegressionDixonColesModel(RDC_FEAT, xi=0.00038).fit(train)
print('Ensemble fit complete.')
print(f'  RegDC coeffs:\n{rdc.coefficients().round(3).to_string(index=False)}')

Ensemble fit complete.
  RegDC coeffs:
feature  w_attack  w_defense
    elo     0.160      0.129
   logv     0.173      0.197
   fpts     0.292      0.287


## Per-team feature snapshot for the 48 WC 2026 teams

In [4]:
all_teams = sum(WC_2026_GROUPS.values(), [])
TEAM_SNAPSHOT = {}
for t in all_teams:
    home_recent = df[df['home_team']==t].sort_values('date').tail(1)
    away_recent = df[df['away_team']==t].sort_values('date').tail(1)
    if not home_recent.empty and (away_recent.empty or home_recent['date'].iloc[0] >= away_recent['date'].iloc[0]):
        r = home_recent.iloc[0]; side='home'
    elif not away_recent.empty:
        r = away_recent.iloc[0]; side='away'
    else:
        raise ValueError(f'No data for {t}')
    TEAM_SNAPSHOT[t] = {
        'elo_pre': r[f'{side}_elo_pre'],
        'top_n_value_eur': r[f'{side}_top_n_value_eur'],
        'outfield_mean_age': r[f'{side}_outfield_mean_age'],
        'top1_share': r[f'{side}_top1_share'],
        'fifa_points': r[f'{side}_fifa_points'] if pd.notna(r[f'{side}_fifa_points']) else 0,
        'last_date': r['date'].strftime('%Y-%m-%d'),
    }
print(f'Snapshots built for {len(TEAM_SNAPSHOT)} teams')
print(f'Sample (Spain): {TEAM_SNAPSHOT["Spain"]}')

Snapshots built for 48 teams
Sample (Spain): {'elo_pre': np.float64(1904.0510466525968), 'top_n_value_eur': np.float64(1435000000.0), 'outfield_mean_age': np.float64(23.74962478249755), 'top1_share': np.float64(0.1393728222996515), 'fifa_points': np.float64(1877.178865), 'last_date': '2025-11-18'}


## Ensemble matchup function + Reg-DC shim for the bracket sim

In [5]:
class RegDCShim:
    def __init__(self, rdc, snapshot):
        self.rdc = rdc
        self.attack, self.defense = {}, {}
        for t, s in snapshot.items():
            x = np.array([s['elo_pre']/100.0,
                          np.log10(max(s['top_n_value_eur'] or FLOOR, FLOOR)),
                          (s['fifa_points'] or 0)/1000.0])
            x_c = x - rdc.feature_mean_
            self.attack[t]  = float(x_c @ rdc.w_a)
            self.defense[t] = float(x_c @ rdc.w_b)
        self.home_adv = rdc.home_adv; self.rho = rdc.rho; self.max_goals = rdc.max_goals
    def _rates(self, h, a, neutral=False):
        gamma = 0.0 if neutral else self.home_adv
        lh = float(np.exp(self.attack[h] - self.defense[a] + gamma))
        la = float(np.exp(self.attack[a] - self.defense[h]))
        return lh, la
    def score_matrix(self, h, a, neutral=False):
        lh, la = self._rates(h, a, neutral)
        x = np.arange(self.max_goals+1)
        m = np.outer(poisson.pmf(x, lh), poisson.pmf(x, la)).copy()
        m[0,0] *= 1 - lh*la*self.rho; m[0,1] *= 1 + lh*self.rho
        m[1,0] *= 1 + la*self.rho;    m[1,1] *= 1 - self.rho
        return m

shim = RegDCShim(rdc, TEAM_SNAPSHOT)

def _safe(v, default=0.0):
    '''Treat NaN/None as default. Python `or` doesn't catch NaN since NaN is truthy.'''
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return default
    return v

def matchup_diff_features(home: str, away: str) -> np.ndarray:
    h, a = TEAM_SNAPSHOT[home], TEAM_SNAPSHOT[away]
    elo_diff = (_safe(h['elo_pre'], 1500) - _safe(a['elo_pre'], 1500)) / 400.0
    log_val_diff = np.log10(max(_safe(h['top_n_value_eur'], FLOOR), FLOOR) /
                            max(_safe(a['top_n_value_eur'], FLOOR), FLOOR))
    age_diff   = _safe(h['outfield_mean_age']) - _safe(a['outfield_mean_age'])
    top1_diff  = _safe(h['top1_share'])        - _safe(a['top1_share'])
    fpts_diff  = (_safe(h['fifa_points'])      - _safe(a['fifa_points'])) / 1000.0
    return np.array([elo_diff, log_val_diff, age_diff, top1_diff, fpts_diff])

def ensemble_predict(home: str, away: str):
    '''Returns (p_away, p_draw, p_home) averaged across MLP+XGB+CB.'''
    x = matchup_diff_features(home, away).reshape(1, -1)
    xs = scaler.transform(x)
    p_mlp = mlp.predict_proba(xs)[0]
    p_xgb = xgb_clf.predict_proba(x)[0]
    p_cb  = cb_clf.predict_proba(x)[0]
    # Align class order (each model emits in self.classes_ order)
    def reorder(p, classes): return np.array([p[list(classes).index(c)] for c in (0,1,2)])
    p_mlp = reorder(p_mlp, mlp.classes_); p_xgb = reorder(p_xgb, xgb_clf.classes_); p_cb = reorder(p_cb, cb_clf.classes_.flatten())
    avg = (p_mlp + p_xgb + p_cb) / 3.0
    return tuple(avg / avg.sum())  # (p_away, p_draw, p_home)

# Quick sanity check
for (h, a) in [('Spain','Argentina'), ('Brazil','France'), ('Mexico','South Korea'), ('Iran','Belgium')]:
    p = ensemble_predict(h, a)
    print(f'  {h:12s} vs {a:12s}   P(away)={p[0]:.3f}  P(draw)={p[1]:.3f}  P(home)={p[2]:.3f}')

  Spain        vs Argentina      P(away)=0.219  P(draw)=0.270  P(home)=0.511
  Brazil       vs France         P(away)=0.497  P(draw)=0.302  P(home)=0.201
  Mexico       vs South Korea    P(away)=0.322  P(draw)=0.286  P(home)=0.391
  Iran         vs Belgium        P(away)=0.491  P(draw)=0.250  P(home)=0.259


## Run hybrid bracket sim

In [6]:
results, bracket_data = simulate_tournament_48(WC_2026_GROUPS, shim, n_sims=5000, n_group_sims=2000,
                                  seed=0, outcome_model=ensemble_predict, return_bracket=True)
pd.options.display.float_format = '{:.3f}'.format
print('Top 12 with ensemble outcome model:')
print(results.head(12).to_string(index=False))

Top 12 with ensemble outcome model:
       team group  p_R32  p_R16  p_QF  p_SF  p_final  p_winner
      Spain     H  1.000  0.848 0.655 0.567    0.402     0.298
    England     L  0.991  0.807 0.588 0.408    0.277     0.144
     France     I  0.992  0.771 0.526 0.369    0.200     0.119
  Argentina     J  0.987  0.654 0.514 0.344    0.196     0.103
   Portugal     K  1.000  0.772 0.508 0.298    0.153     0.066
Netherlands     F  0.957  0.570 0.368 0.193    0.093     0.044
     Brazil     C  0.962  0.559 0.342 0.162    0.085     0.034
    Germany     E  0.999  0.556 0.256 0.139    0.059     0.024
    Belgium     G  0.954  0.638 0.360 0.135    0.057     0.021
    Morocco     C  0.921  0.466 0.277 0.130    0.055     0.019
      Japan     F  0.920  0.476 0.278 0.113    0.045     0.016
     Norway     I  0.922  0.480 0.223 0.082    0.033     0.012


## Per-matchup probabilities (all 48 × 47 / 2 = 1128 possible matchups)

Cached for the web visualization.

In [7]:
matchup_probs = []
for h, a in combinations(all_teams, 2):
    p_h_to_a = ensemble_predict(h, a)        # if h is home
    p_a_to_h = ensemble_predict(a, h)        # if a is home (reversed perspective)
    # Symmetric (neutral venue) avg
    p_home_wins = (p_h_to_a[2] + p_a_to_h[0]) / 2
    p_away_wins = (p_h_to_a[0] + p_a_to_h[2]) / 2
    p_draw      = (p_h_to_a[1] + p_a_to_h[1]) / 2
    s = p_home_wins + p_draw + p_away_wins
    matchup_probs.append({'team_a': h, 'team_b': a,
                          'p_a_wins': float(p_home_wins/s),
                          'p_draw':   float(p_draw/s),
                          'p_b_wins': float(p_away_wins/s)})
matchup_df = pd.DataFrame(matchup_probs)
print(f'{len(matchup_df)} neutral matchups computed')
print(matchup_df.head(5).round(3).to_string(index=False))

1128 neutral matchups computed
team_a                 team_b  p_a_wins  p_draw  p_b_wins
Mexico           South Africa     0.649   0.224     0.127
Mexico            South Korea     0.410   0.278     0.312
Mexico         Czech Republic     0.533   0.285     0.182
Mexico                 Canada     0.406   0.293     0.301
Mexico Bosnia and Herzegovina     0.770   0.191     0.038


## Export predictions.json for the web viz

In [8]:
from pathlib import Path
out_dir = Path('../viz/public')
out_dir.mkdir(parents=True, exist_ok=True)

payload = {
    'generated_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'model': 'Reg-DC (group stage scoring) + ensemble (MLP+XGB+CatBoost, knockout outcomes)',
    'n_sims': 5000,
    'groups': WC_2026_GROUPS,
    'team_summary': results.to_dict(orient='records'),
    'matchups': matchup_df.to_dict(orient='records'),
    'bracket': bracket_data,
    'team_snapshot': {t: {k: (None if pd.isna(v) else (float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v)) for k, v in s.items()} for t, s in TEAM_SNAPSHOT.items()},
}
with open(out_dir/'predictions.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=1, default=str)
print(f'Wrote {out_dir/"predictions.json"}')
print(f'  team_summary: {len(payload["team_summary"])} teams')
print(f'  matchups: {len(payload["matchups"])} matchups')

Wrote ..\viz\public\predictions.json
  team_summary: 48 teams
  matchups: 1128 matchups
